In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [2]:
!ls /content/drive/MyDrive/ZC-CSA

Resnet.pt  running.ipynb  selected_samples_for_attack_cifar10.pt


# On class 0,1

In [ ]:
import os
import time
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd

# -------------------------
# CIFAR10 setup
# -------------------------
img_shape = (3, 32, 32)
dim = int(np.prod(img_shape))

# Pixel bounds in RAW space:
# We'll assume samples are uint8-like in [0,255] unless detected otherwise.
final_lower_bound = 0.0
final_upper_bound = 255.0

# Perturbation bounds (tune as you like for CIFAR):
# These are per-pixel deltas in raw [0..255] space.
raw_lower_bound = -20.0
raw_upper_bound = 20.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

raw_lower   = torch.tensor(raw_lower_bound,   dtype=torch.float32, device=device)
raw_upper   = torch.tensor(raw_upper_bound,   dtype=torch.float32, device=device)
final_lower = torch.tensor(final_lower_bound, dtype=torch.float32, device=device)
final_upper = torch.tensor(final_upper_bound, dtype=torch.float32, device=device)

# CIFAR10 normalization (match your training!)
CIFAR10_MEAN = torch.tensor([0.4914, 0.4822, 0.4465], device=device).view(1,3,1,1)
CIFAR10_STD  = torch.tensor([0.2470, 0.2435, 0.2616], device=device).view(1,3,1,1)

# Always CNN mode for ResNet
CNN_MODE = True


def clamp_tensor(x: torch.Tensor, lower: torch.Tensor, upper: torch.Tensor) -> torch.Tensor:
    return torch.max(torch.min(x, upper), lower)


def to_model_input(x_raw: torch.Tensor) -> torch.Tensor:
    """
    x_raw: [1,3,32,32] in raw pixel space.
    Returns normalized tensor suitable for ResNet.
    """
    x_clip = clamp_tensor(x_raw, final_lower, final_upper)
    x_01 = x_clip / final_upper  # [0,1]
    x_norm = (x_01 - CIFAR10_MEAN) / CIFAR10_STD
    return x_norm


def calculate_adversarial_loss(model: nn.Module, x_raw_single: torch.Tensor, y_single: torch.Tensor, eps: float = 1e-9) -> torch.Tensor:
    """
    x_raw_single: [1,3,32,32] raw pixels
    y_single: scalar tensor or shape [1]
    """
    if y_single.ndim == 0:
        y_single = y_single.view(1)

    x_in = to_model_input(x_raw_single)
    logits = model(x_in)

    log_probs = F.log_softmax(logits, dim=1)
    ce = -log_probs[0, y_single.item()]
    return torch.clamp(ce, min=eps)


def save_cifar_uint8_rgb(img_chw: np.ndarray, path: str) -> None:
    """
    img_chw: (3,32,32) float or uint8 in [0,255]
    Saves as PNG. Converts RGB->BGR for OpenCV.
    """
    img = np.clip(np.round(img_chw), 0, 255).astype(np.uint8)
    img_hwc = np.transpose(img, (1,2,0))          # HWC RGB
    img_bgr = img_hwc[..., ::-1]                  # RGB->BGR
    cv2.imwrite(path, img_bgr)


def load_model(model_path: str) -> nn.Module:
    # Try TorchScript first
    try:
        m = torch.jit.load(model_path, map_location=device)
        m.eval().to(device)
        print(f"Loaded TorchScript model from {model_path}")
        return m
    except Exception as e1:
        # Fallback: regular torch.load (could be full model or state_dict)
        ckpt = torch.load(model_path, map_location=device)
        if isinstance(ckpt, nn.Module):
            m = ckpt
            m.eval().to(device)
            print(f"Loaded nn.Module from torch.load: {model_path}")
            return m
        raise RuntimeError(
            f"Could not load model as TorchScript or nn.Module. "
            f"If this is a state_dict, you must rebuild ResNet18 and load_state_dict.\n"
            f"TorchScript error: {e1}"
        )


def load_samples(samples_path: str):
    obj = torch.load(samples_path, map_location="cpu")

    # Debugging: Print type and content of obj
    print(f"Loaded object type: {type(obj)}")
    if isinstance(obj, dict):
        print(f"Loaded dictionary keys: {obj.keys()}")
        for k, v in obj.items():
            if torch.is_tensor(v):
                print(f"  Key '{k}': Tensor shape {v.shape}, dtype {v.dtype}")
            else:
                print(f"  Key '{k}': Type {type(v)}")
    elif isinstance(obj, (tuple, list)):
        print(f"Loaded tuple/list length: {len(obj)}")
        for i, item in enumerate(obj):
            if torch.is_tensor(item):
                print(f"  Item {i}: Tensor shape {item.shape}, dtype {item.dtype}")
            else:
                print(f"  Item {i}: Type {type(item)}")
    else:
        print(f"Loaded object value (first 100 chars): {str(obj)[:100]}")


    # Common patterns:
    if isinstance(obj, (tuple, list)) and len(obj) == 2 and torch.is_tensor(obj[0]) and torch.is_tensor(obj[1]):
        X, y = obj
    elif isinstance(obj, dict) and ("images" in obj and "labels" in obj):
        X, y = obj["images"], obj["labels"]
    elif isinstance(obj, dict) and ("X" in obj and "y" in obj):
        X, y = obj["X"], obj["y"]
    else:
        raise ValueError("Unrecognized samples format. Expected (X,y) tuple or {'X':..., 'y':...} dict.")

    # Ensure shapes: X [N,3,32,32], y [N]
    if X.ndim == 2 and X.shape[1] == dim:
        X = X.view(-1, *img_shape)
    if X.ndim != 4:
        raise ValueError(f"X must be [N,3,32,32]. Got {X.shape}")
    if y.ndim != 1:
        y = y.view(-1)

    # Detect whether X is already 0..255 or 0..1
    Xf = X.float()
    maxv = float(Xf.max().item())
    if maxv <= 1.5:
        # looks like [0,1]
        Xf = Xf * 255.0
        print("Detected samples in [0,1]; converted to raw [0,255].")
    else:
        print("Detected samples in raw-like scale; using as [0,255].")

    return Xf, y.long()


class CuckooSearchSingleSample:
    def __init__(self,
                 n_nests: int,
                 n_iterations: int,
                 dim: int,
                 model: nn.Module,
                 X_sample_chw: torch.Tensor,   # [3,32,32] raw
                 y_sample: torch.Tensor,       # scalar or [1]
                 p_a: float = 0.5,
                 lambda_param: float = 1e8,
                 step_size: float = 2.0,
                 step_decay: float = 0.98):
        self.n_nests = n_nests
        self.n_iterations = n_iterations
        self.dim = dim
        self.model = model
        self.X = X_sample_chw.unsqueeze(0).to(device)  # [1,3,32,32]
        self.y = y_sample.to(device)
        self.p_a = p_a
        self.lambda_param = lambda_param
        self.step_size = step_size
        self.step_decay = step_decay

        self.nests = torch.zeros((n_nests, dim), dtype=torch.float32, device=device)
        self.fitness = torch.empty(n_nests, dtype=torch.float32, device=device)

        for i in range(n_nests):
            self.fitness[i] = self._fitness(self.nests[i])

        best_idx = torch.argmin(self.fitness)
        self.best_nest = self.nests[best_idx].clone()
        self.best_fitness = float(self.fitness[best_idx].item())

    def _cauchy_flight(self) -> torch.Tensor:
        dist = torch.distributions.Cauchy(loc=torch.tensor(0.0, device=device),
                                          scale=torch.tensor(self.step_size, device=device))
        return dist.sample((self.dim,))

    def _new_solution(self, curr: torch.Tensor) -> torch.Tensor:
        cand = curr + self._cauchy_flight()
        return clamp_tensor(cand, raw_lower, raw_upper)

    def _fitness(self, perturb_flat: torch.Tensor) -> float:
        p = clamp_tensor(perturb_flat, raw_lower, raw_upper).view(1, *img_shape)  # [1,3,32,32]
        x_adv_raw = self.X + p
        loss = calculate_adversarial_loss(self.model, x_adv_raw, self.y).item()
        mag = torch.norm(p).item()
        return mag - self.lambda_param * loss

    def optimize(self) -> None:
        for it in range(1, self.n_iterations + 1):
            for i in range(self.n_nests):
                cand = self._new_solution(self.nests[i])
                cand_fit = self._fitness(cand)
                if cand_fit < self.fitness[i]:
                    self.nests[i] = cand
                    self.fitness[i] = cand_fit
                    if cand_fit < self.best_fitness:
                        self.best_fitness = cand_fit
                        self.best_nest = cand.clone()

            k = int(self.n_nests * self.p_a)
            if k > 0:
                worst_idx = torch.argsort(self.fitness)[-k:]
                for idx in worst_idx:
                    new_rand = torch.empty((self.dim,), device=device).uniform_(float(raw_lower), float(raw_upper))
                    self.nests[idx] = new_rand
                    self.fitness[idx] = self._fitness(new_rand)

            if it in {1, self.n_iterations} or it % 10 == 0:
                print(f"    iter {it}/{self.n_iterations} — best fitness {self.best_fitness:.5f}")

            self.step_size *= self.step_decay

    def get_best_solution(self) -> torch.Tensor:
        return self.best_nest


def main():
    MODEL_PATH = "/content/drive/MyDrive/ZC-CSA/Resnet.pt"
    SAMPLES_PATH = "/content/drive/MyDrive/ZC-CSA/selected_samples_for_attack_cifar10.pt"

    model = load_model(MODEL_PATH)
    X_full, y_full = load_samples(SAMPLES_PATH)

    # outputs
    base_dir = "/content/drive/MyDrive/including_cifar10"
    orig_dir = os.path.join(base_dir, "Original")
    adv_dir  = os.path.join(base_dir, "Adversarial")
    pert_dir = os.path.join(base_dir, "Perturbations")
    for d in (orig_dir, adv_dir, pert_dir):
        os.makedirs(d, exist_ok=True)

    # hyperparams (you will likely tune lambda_param / step_size / bounds for CIFAR)
    n_nests = 58
    n_iterations = 100
    p_a = 0.5
    lambda_param = 1e8
    step_size = 2.0
    step_decay = 0.98

    results = []
    processed = 0
    success_count = 0

    X_full = X_full.to(device)
    y_full = y_full.to(device)

    tic = time.time()
    for cls in range(2):
        idxs = (y_full == cls).nonzero(as_tuple=True)[0]
        if idxs.numel() == 0:
            print(f"No samples for class {cls}; skipping.")
            continue

        print(f"\n── Class {cls}: {idxs.numel()} samples ──")
        for j, gidx in enumerate(idxs):
            x = X_full[gidx]              # [3,32,32] raw
            y = y_full[gidx]              # scalar
            gidx_int = int(gidx.item())

            print(f"  sample {j+1}/{idxs.numel()} (global {gidx_int})")

            save_cifar_uint8_rgb(x.detach().cpu().numpy(),
                                 os.path.join(orig_dir, f"{gidx_int}_lbl{cls}.png"))

            css = CuckooSearchSingleSample(
                n_nests=n_nests,
                n_iterations=n_iterations,
                dim=dim,
                model=model,
                X_sample_chw=x,
                y_sample=y,
                p_a=p_a,
                lambda_param=lambda_param,
                step_size=step_size,
                step_decay=step_decay
            )
            css.optimize()
            delta_flat = css.get_best_solution()
            delta = delta_flat.view(*img_shape)

            x_adv_raw = clamp_tensor(x + delta, final_lower, final_upper)

            with torch.no_grad():
                pred = torch.argmax(model(to_model_input(x_adv_raw.unsqueeze(0))), dim=1).item()

            l2 = float(torch.norm(delta).item())
            miscls = int(pred != int(y.item()))

            # save perturbation
            delta_np = delta.detach().cpu().numpy()
            np.save(os.path.join(pert_dir, f"{gidx_int}_delta.npy"), delta_np)

            # visualize delta (map [-B,+B] -> [0,255])
            B = max(abs(raw_lower_bound), abs(raw_upper_bound))
            delta_vis = (np.clip(delta_np, -B, B) + B) / (2 * B) * 255.0
            save_cifar_uint8_rgb(delta_vis, os.path.join(pert_dir, f"{gidx_int}_delta_vis.png"))

            if miscls:
                success_count += 1
                save_cifar_uint8_rgb(x_adv_raw.detach().cpu().numpy(),
                                     os.path.join(adv_dir, f"{gidx_int}_t{cls}_p{pred}_m{l2:.2f}.png"))

            results.append({
                "class": cls,
                "index": gidx_int,
                "true": int(y.item()),
                "pred": int(pred),
                "misclassified": bool(miscls),
                "perturb_norm_l2": l2
            })
            processed += 1

    df = pd.DataFrame(results)
    os.makedirs(base_dir, exist_ok=True)
    csv_path = os.path.join(base_dir, "0_1_zc_csa_cifar10_results.csv")
    df.to_csv(csv_path, index=False)

    stats = (df.groupby(["class", "misclassified"])
               .size()
               .unstack(fill_value=0)
               .rename(columns={False: "correct", True: "misclassified"}))
    if "misclassified" not in stats.columns:
        stats["misclassified"] = 0
    if "correct" not in stats.columns:
        stats["correct"] = 0
    stats["total"] = stats["correct"] + stats["misclassified"]
    stats["correct_%"] = (stats["correct"]/stats["total"]*100).round(2)
    stats["miscls_%"]  = (stats["misclassified"]/stats["total"]*100).round(2)

    stats_path = os.path.join(base_dir, "0_1_zc_csa_cifar10_stats_per_class.csv")
    stats.to_csv(stats_path)

    toc = time.time()
    print("\n===== SUMMARY =====")
    print(f"Processed samples        : {processed}")
    print(f"Successful attacks       : {success_count}")
    print(f"Results CSV              : {csv_path}")
    print(f"Per-class stats CSV      : {stats_path}")
    print(f"Total time               : {toc - tic:.2f} s\n")
    print(stats.to_string())


if __name__ == "__main__":
    main()

Loaded TorchScript model from /content/drive/MyDrive/ZC-CSA/Resnet.pt
Loaded object type: <class 'dict'>
Loaded dictionary keys: dict_keys(['images', 'labels'])
  Key 'images': Tensor shape torch.Size([1000, 3, 32, 32]), dtype torch.float32
  Key 'labels': Tensor shape torch.Size([1000]), dtype torch.int64
Detected samples in [0,1]; converted to raw [0,255].

── Class 0: 100 samples ──
  sample 1/100 (global 0)
    iter 1/100 — best fitness -395142602.13510
    iter 10/100 — best fitness -460118960.81763
    iter 20/100 — best fitness -482050927.87457
    iter 30/100 — best fitness -489049118.67059
    iter 40/100 — best fitness -500783296.08612
    iter 50/100 — best fitness -504214899.42102
    iter 60/100 — best fitness -504214899.42102
    iter 70/100 — best fitness -509960214.56635
    iter 80/100 — best fitness -509960214.56635
    iter 90/100 — best fitness -517010913.75366
    iter 100/100 — best fitness -519051492.07605
  sample 2/100 (global 1)
    iter 1/100 — best fitness -

# On classes 2,3

In [ ]:
import os
import time
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd

# -------------------------
# CIFAR10 setup
# -------------------------
img_shape = (3, 32, 32)
dim = int(np.prod(img_shape))

# Pixel bounds in RAW space:
# We'll assume samples are uint8-like in [0,255] unless detected otherwise.
final_lower_bound = 0.0
final_upper_bound = 255.0

# Perturbation bounds (tune as you like for CIFAR):
# These are per-pixel deltas in raw [0..255] space.
raw_lower_bound = -20.0
raw_upper_bound = 20.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

raw_lower   = torch.tensor(raw_lower_bound,   dtype=torch.float32, device=device)
raw_upper   = torch.tensor(raw_upper_bound,   dtype=torch.float32, device=device)
final_lower = torch.tensor(final_lower_bound, dtype=torch.float32, device=device)
final_upper = torch.tensor(final_upper_bound, dtype=torch.float32, device=device)

# CIFAR10 normalization (match your training!)
CIFAR10_MEAN = torch.tensor([0.4914, 0.4822, 0.4465], device=device).view(1,3,1,1)
CIFAR10_STD  = torch.tensor([0.2470, 0.2435, 0.2616], device=device).view(1,3,1,1)

# Always CNN mode for ResNet
CNN_MODE = True


def clamp_tensor(x: torch.Tensor, lower: torch.Tensor, upper: torch.Tensor) -> torch.Tensor:
    return torch.max(torch.min(x, upper), lower)


def to_model_input(x_raw: torch.Tensor) -> torch.Tensor:
    """
    x_raw: [1,3,32,32] in raw pixel space.
    Returns normalized tensor suitable for ResNet.
    """
    x_clip = clamp_tensor(x_raw, final_lower, final_upper)
    x_01 = x_clip / final_upper  # [0,1]
    x_norm = (x_01 - CIFAR10_MEAN) / CIFAR10_STD
    return x_norm


def calculate_adversarial_loss(model: nn.Module, x_raw_single: torch.Tensor, y_single: torch.Tensor, eps: float = 1e-9) -> torch.Tensor:
    """
    x_raw_single: [1,3,32,32] raw pixels
    y_single: scalar tensor or shape [1]
    """
    if y_single.ndim == 0:
        y_single = y_single.view(1)

    x_in = to_model_input(x_raw_single)
    logits = model(x_in)

    log_probs = F.log_softmax(logits, dim=1)
    ce = -log_probs[0, y_single.item()]
    return torch.clamp(ce, min=eps)


def save_cifar_uint8_rgb(img_chw: np.ndarray, path: str) -> None:
    """
    img_chw: (3,32,32) float or uint8 in [0,255]
    Saves as PNG. Converts RGB->BGR for OpenCV.
    """
    img = np.clip(np.round(img_chw), 0, 255).astype(np.uint8)
    img_hwc = np.transpose(img, (1,2,0))          # HWC RGB
    img_bgr = img_hwc[..., ::-1]                  # RGB->BGR
    cv2.imwrite(path, img_bgr)


def load_model(model_path: str) -> nn.Module:
    # Try TorchScript first
    try:
        m = torch.jit.load(model_path, map_location=device)
        m.eval().to(device)
        print(f"Loaded TorchScript model from {model_path}")
        return m
    except Exception as e1:
        # Fallback: regular torch.load (could be full model or state_dict)
        ckpt = torch.load(model_path, map_location=device)
        if isinstance(ckpt, nn.Module):
            m = ckpt
            m.eval().to(device)
            print(f"Loaded nn.Module from torch.load: {model_path}")
            return m
        raise RuntimeError(
            f"Could not load model as TorchScript or nn.Module. "
            f"If this is a state_dict, you must rebuild ResNet18 and load_state_dict.\n"
            f"TorchScript error: {e1}"
        )


def load_samples(samples_path: str):
    obj = torch.load(samples_path, map_location="cpu")

    # Debugging: Print type and content of obj
    print(f"Loaded object type: {type(obj)}")
    if isinstance(obj, dict):
        print(f"Loaded dictionary keys: {obj.keys()}")
        for k, v in obj.items():
            if torch.is_tensor(v):
                print(f"  Key '{k}': Tensor shape {v.shape}, dtype {v.dtype}")
            else:
                print(f"  Key '{k}': Type {type(v)}")
    elif isinstance(obj, (tuple, list)):
        print(f"Loaded tuple/list length: {len(obj)}")
        for i, item in enumerate(obj):
            if torch.is_tensor(item):
                print(f"  Item {i}: Tensor shape {item.shape}, dtype {item.dtype}")
            else:
                print(f"  Item {i}: Type {type(item)}")
    else:
        print(f"Loaded object value (first 100 chars): {str(obj)[:100]}")


    # Common patterns:
    if isinstance(obj, (tuple, list)) and len(obj) == 2 and torch.is_tensor(obj[0]) and torch.is_tensor(obj[1]):
        X, y = obj
    elif isinstance(obj, dict) and ("images" in obj and "labels" in obj):
        X, y = obj["images"], obj["labels"]
    elif isinstance(obj, dict) and ("X" in obj and "y" in obj):
        X, y = obj["X"], obj["y"]
    else:
        raise ValueError("Unrecognized samples format. Expected (X,y) tuple or {'X':..., 'y':...} dict.")

    # Ensure shapes: X [N,3,32,32], y [N]
    if X.ndim == 2 and X.shape[1] == dim:
        X = X.view(-1, *img_shape)
    if X.ndim != 4:
        raise ValueError(f"X must be [N,3,32,32]. Got {X.shape}")
    if y.ndim != 1:
        y = y.view(-1)

    # Detect whether X is already 0..255 or 0..1
    Xf = X.float()
    maxv = float(Xf.max().item())
    if maxv <= 1.5:
        # looks like [0,1]
        Xf = Xf * 255.0
        print("Detected samples in [0,1]; converted to raw [0,255].")
    else:
        print("Detected samples in raw-like scale; using as [0,255].")

    return Xf, y.long()


class CuckooSearchSingleSample:
    def __init__(self,
                 n_nests: int,
                 n_iterations: int,
                 dim: int,
                 model: nn.Module,
                 X_sample_chw: torch.Tensor,   # [3,32,32] raw
                 y_sample: torch.Tensor,       # scalar or [1]
                 p_a: float = 0.5,
                 lambda_param: float = 1e8,
                 step_size: float = 2.0,
                 step_decay: float = 0.98):
        self.n_nests = n_nests
        self.n_iterations = n_iterations
        self.dim = dim
        self.model = model
        self.X = X_sample_chw.unsqueeze(0).to(device)  # [1,3,32,32]
        self.y = y_sample.to(device)
        self.p_a = p_a
        self.lambda_param = lambda_param
        self.step_size = step_size
        self.step_decay = step_decay

        self.nests = torch.zeros((n_nests, dim), dtype=torch.float32, device=device)
        self.fitness = torch.empty(n_nests, dtype=torch.float32, device=device)

        for i in range(n_nests):
            self.fitness[i] = self._fitness(self.nests[i])

        best_idx = torch.argmin(self.fitness)
        self.best_nest = self.nests[best_idx].clone()
        self.best_fitness = float(self.fitness[best_idx].item())

    def _cauchy_flight(self) -> torch.Tensor:
        dist = torch.distributions.Cauchy(loc=torch.tensor(0.0, device=device),
                                          scale=torch.tensor(self.step_size, device=device))
        return dist.sample((self.dim,))

    def _new_solution(self, curr: torch.Tensor) -> torch.Tensor:
        cand = curr + self._cauchy_flight()
        return clamp_tensor(cand, raw_lower, raw_upper)

    def _fitness(self, perturb_flat: torch.Tensor) -> float:
        p = clamp_tensor(perturb_flat, raw_lower, raw_upper).view(1, *img_shape)  # [1,3,32,32]
        x_adv_raw = self.X + p
        loss = calculate_adversarial_loss(self.model, x_adv_raw, self.y).item()
        mag = torch.norm(p).item()
        return mag - self.lambda_param * loss

    def optimize(self) -> None:
        for it in range(1, self.n_iterations + 1):
            for i in range(self.n_nests):
                cand = self._new_solution(self.nests[i])
                cand_fit = self._fitness(cand)
                if cand_fit < self.fitness[i]:
                    self.nests[i] = cand
                    self.fitness[i] = cand_fit
                    if cand_fit < self.best_fitness:
                        self.best_fitness = cand_fit
                        self.best_nest = cand.clone()

            k = int(self.n_nests * self.p_a)
            if k > 0:
                worst_idx = torch.argsort(self.fitness)[-k:]
                for idx in worst_idx:
                    new_rand = torch.empty((self.dim,), device=device).uniform_(float(raw_lower), float(raw_upper))
                    self.nests[idx] = new_rand
                    self.fitness[idx] = self._fitness(new_rand)

            if it in {1, self.n_iterations} or it % 10 == 0:
                print(f"    iter {it}/{self.n_iterations} — best fitness {self.best_fitness:.5f}")

            self.step_size *= self.step_decay

    def get_best_solution(self) -> torch.Tensor:
        return self.best_nest


def main():
    MODEL_PATH = "/content/drive/MyDrive/ZC-CSA/Resnet.pt"
    SAMPLES_PATH = "/content/drive/MyDrive/ZC-CSA/selected_samples_for_attack_cifar10.pt"

    model = load_model(MODEL_PATH)
    X_full, y_full = load_samples(SAMPLES_PATH)

    # outputs
    base_dir = "/content/drive/MyDrive/including_cifar10"
    orig_dir = os.path.join(base_dir, "Original")
    adv_dir  = os.path.join(base_dir, "Adversarial")
    pert_dir = os.path.join(base_dir, "Perturbations")
    for d in (orig_dir, adv_dir, pert_dir):
        os.makedirs(d, exist_ok=True)

    # hyperparams (you will likely tune lambda_param / step_size / bounds for CIFAR)
    n_nests = 58
    n_iterations = 100
    p_a = 0.5
    lambda_param = 1e8
    step_size = 2.0
    step_decay = 0.98

    results = []
    processed = 0
    success_count = 0

    X_full = X_full.to(device)
    y_full = y_full.to(device)

    tic = time.time()
    for cls in range(2,4):
        idxs = (y_full == cls).nonzero(as_tuple=True)[0]
        if idxs.numel() == 0:
            print(f"No samples for class {cls}; skipping.")
            continue

        print(f"\n── Class {cls}: {idxs.numel()} samples ──")
        for j, gidx in enumerate(idxs):
            x = X_full[gidx]              # [3,32,32] raw
            y = y_full[gidx]              # scalar
            gidx_int = int(gidx.item())

            print(f"  sample {j+1}/{idxs.numel()} (global {gidx_int})")

            save_cifar_uint8_rgb(x.detach().cpu().numpy(),
                                 os.path.join(orig_dir, f"{gidx_int}_lbl{cls}.png"))

            css = CuckooSearchSingleSample(
                n_nests=n_nests,
                n_iterations=n_iterations,
                dim=dim,
                model=model,
                X_sample_chw=x,
                y_sample=y,
                p_a=p_a,
                lambda_param=lambda_param,
                step_size=step_size,
                step_decay=step_decay
            )
            css.optimize()
            delta_flat = css.get_best_solution()
            delta = delta_flat.view(*img_shape)

            x_adv_raw = clamp_tensor(x + delta, final_lower, final_upper)

            with torch.no_grad():
                pred = torch.argmax(model(to_model_input(x_adv_raw.unsqueeze(0))), dim=1).item()

            l2 = float(torch.norm(delta).item())
            miscls = int(pred != int(y.item()))

            # save perturbation
            delta_np = delta.detach().cpu().numpy()
            np.save(os.path.join(pert_dir, f"{gidx_int}_delta.npy"), delta_np)

            # visualize delta (map [-B,+B] -> [0,255])
            B = max(abs(raw_lower_bound), abs(raw_upper_bound))
            delta_vis = (np.clip(delta_np, -B, B) + B) / (2 * B) * 255.0
            save_cifar_uint8_rgb(delta_vis, os.path.join(pert_dir, f"{gidx_int}_delta_vis.png"))

            if miscls:
                success_count += 1
                save_cifar_uint8_rgb(x_adv_raw.detach().cpu().numpy(),
                                     os.path.join(adv_dir, f"{gidx_int}_t{cls}_p{pred}_m{l2:.2f}.png"))

            results.append({
                "class": cls,
                "index": gidx_int,
                "true": int(y.item()),
                "pred": int(pred),
                "misclassified": bool(miscls),
                "perturb_norm_l2": l2
            })
            processed += 1

    df = pd.DataFrame(results)
    os.makedirs(base_dir, exist_ok=True)
    csv_path = os.path.join(base_dir, "2_3_zc_csa_cifar10_results.csv")
    df.to_csv(csv_path, index=False)

    stats = (df.groupby(["class", "misclassified"])
               .size()
               .unstack(fill_value=0)
               .rename(columns={False: "correct", True: "misclassified"}))
    if "misclassified" not in stats.columns:
        stats["misclassified"] = 0
    if "correct" not in stats.columns:
        stats["correct"] = 0
    stats["total"] = stats["correct"] + stats["misclassified"]
    stats["correct_%"] = (stats["correct"]/stats["total"]*100).round(2)
    stats["miscls_%"]  = (stats["misclassified"]/stats["total"]*100).round(2)

    stats_path = os.path.join(base_dir, "2_3_zc_csa_cifar10_stats_per_class.csv")
    stats.to_csv(stats_path)

    toc = time.time()
    print("\n===== SUMMARY =====")
    print(f"Processed samples        : {processed}")
    print(f"Successful attacks       : {success_count}")
    print(f"Results CSV              : {csv_path}")
    print(f"Per-class stats CSV      : {stats_path}")
    print(f"Total time               : {toc - tic:.2f} s\n")
    print(stats.to_string())


if __name__ == "__main__":
    main()

Loaded TorchScript model from /content/drive/MyDrive/ZC-CSA/Resnet.pt
Loaded object type: <class 'dict'>
Loaded dictionary keys: dict_keys(['images', 'labels'])
  Key 'images': Tensor shape torch.Size([1000, 3, 32, 32]), dtype torch.float32
  Key 'labels': Tensor shape torch.Size([1000]), dtype torch.int64
Detected samples in [0,1]; converted to raw [0,255].

── Class 2: 100 samples ──
  sample 1/100 (global 200)
    iter 1/100 — best fitness -48984903.89014
    iter 10/100 — best fitness -88790411.05562
    iter 20/100 — best fitness -109183392.62231
    iter 30/100 — best fitness -143116113.34113
    iter 40/100 — best fitness -143116113.34113
    iter 50/100 — best fitness -154888420.96527
    iter 60/100 — best fitness -178253358.33932
    iter 70/100 — best fitness -185007181.67996
    iter 80/100 — best fitness -225968696.87585
    iter 90/100 — best fitness -236912208.02820
    iter 100/100 — best fitness -257822900.33661
  sample 2/100 (global 201)
    iter 1/100 — best fitness

# On Classes 4,5

In [ ]:
import os
import time
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd

# -------------------------
# CIFAR10 setup
# -------------------------
img_shape = (3, 32, 32)
dim = int(np.prod(img_shape))

# Pixel bounds in RAW space:
# We'll assume samples are uint8-like in [0,255] unless detected otherwise.
final_lower_bound = 0.0
final_upper_bound = 255.0

# Perturbation bounds (tune as you like for CIFAR):
# These are per-pixel deltas in raw [0..255] space.
raw_lower_bound = -20.0
raw_upper_bound = 20.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

raw_lower   = torch.tensor(raw_lower_bound,   dtype=torch.float32, device=device)
raw_upper   = torch.tensor(raw_upper_bound,   dtype=torch.float32, device=device)
final_lower = torch.tensor(final_lower_bound, dtype=torch.float32, device=device)
final_upper = torch.tensor(final_upper_bound, dtype=torch.float32, device=device)

# CIFAR10 normalization (match your training!)
CIFAR10_MEAN = torch.tensor([0.4914, 0.4822, 0.4465], device=device).view(1,3,1,1)
CIFAR10_STD  = torch.tensor([0.2470, 0.2435, 0.2616], device=device).view(1,3,1,1)

# Always CNN mode for ResNet
CNN_MODE = True


def clamp_tensor(x: torch.Tensor, lower: torch.Tensor, upper: torch.Tensor) -> torch.Tensor:
    return torch.max(torch.min(x, upper), lower)


def to_model_input(x_raw: torch.Tensor) -> torch.Tensor:
    """
    x_raw: [1,3,32,32] in raw pixel space.
    Returns normalized tensor suitable for ResNet.
    """
    x_clip = clamp_tensor(x_raw, final_lower, final_upper)
    x_01 = x_clip / final_upper  # [0,1]
    x_norm = (x_01 - CIFAR10_MEAN) / CIFAR10_STD
    return x_norm


def calculate_adversarial_loss(model: nn.Module, x_raw_single: torch.Tensor, y_single: torch.Tensor, eps: float = 1e-9) -> torch.Tensor:
    """
    x_raw_single: [1,3,32,32] raw pixels
    y_single: scalar tensor or shape [1]
    """
    if y_single.ndim == 0:
        y_single = y_single.view(1)

    x_in = to_model_input(x_raw_single)
    logits = model(x_in)

    log_probs = F.log_softmax(logits, dim=1)
    ce = -log_probs[0, y_single.item()]
    return torch.clamp(ce, min=eps)


def save_cifar_uint8_rgb(img_chw: np.ndarray, path: str) -> None:
    """
    img_chw: (3,32,32) float or uint8 in [0,255]
    Saves as PNG. Converts RGB->BGR for OpenCV.
    """
    img = np.clip(np.round(img_chw), 0, 255).astype(np.uint8)
    img_hwc = np.transpose(img, (1,2,0))          # HWC RGB
    img_bgr = img_hwc[..., ::-1]                  # RGB->BGR
    cv2.imwrite(path, img_bgr)


def load_model(model_path: str) -> nn.Module:
    # Try TorchScript first
    try:
        m = torch.jit.load(model_path, map_location=device)
        m.eval().to(device)
        print(f"Loaded TorchScript model from {model_path}")
        return m
    except Exception as e1:
        # Fallback: regular torch.load (could be full model or state_dict)
        ckpt = torch.load(model_path, map_location=device)
        if isinstance(ckpt, nn.Module):
            m = ckpt
            m.eval().to(device)
            print(f"Loaded nn.Module from torch.load: {model_path}")
            return m
        raise RuntimeError(
            f"Could not load model as TorchScript or nn.Module. "
            f"If this is a state_dict, you must rebuild ResNet18 and load_state_dict.\n"
            f"TorchScript error: {e1}"
        )


def load_samples(samples_path: str):
    obj = torch.load(samples_path, map_location="cpu")

    # Debugging: Print type and content of obj
    print(f"Loaded object type: {type(obj)}")
    if isinstance(obj, dict):
        print(f"Loaded dictionary keys: {obj.keys()}")
        for k, v in obj.items():
            if torch.is_tensor(v):
                print(f"  Key '{k}': Tensor shape {v.shape}, dtype {v.dtype}")
            else:
                print(f"  Key '{k}': Type {type(v)}")
    elif isinstance(obj, (tuple, list)):
        print(f"Loaded tuple/list length: {len(obj)}")
        for i, item in enumerate(obj):
            if torch.is_tensor(item):
                print(f"  Item {i}: Tensor shape {item.shape}, dtype {item.dtype}")
            else:
                print(f"  Item {i}: Type {type(item)}")
    else:
        print(f"Loaded object value (first 100 chars): {str(obj)[:100]}")


    # Common patterns:
    if isinstance(obj, (tuple, list)) and len(obj) == 2 and torch.is_tensor(obj[0]) and torch.is_tensor(obj[1]):
        X, y = obj
    elif isinstance(obj, dict) and ("images" in obj and "labels" in obj):
        X, y = obj["images"], obj["labels"]
    elif isinstance(obj, dict) and ("X" in obj and "y" in obj):
        X, y = obj["X"], obj["y"]
    else:
        raise ValueError("Unrecognized samples format. Expected (X,y) tuple or {'X':..., 'y':...} dict.")

    # Ensure shapes: X [N,3,32,32], y [N]
    if X.ndim == 2 and X.shape[1] == dim:
        X = X.view(-1, *img_shape)
    if X.ndim != 4:
        raise ValueError(f"X must be [N,3,32,32]. Got {X.shape}")
    if y.ndim != 1:
        y = y.view(-1)

    # Detect whether X is already 0..255 or 0..1
    Xf = X.float()
    maxv = float(Xf.max().item())
    if maxv <= 1.5:
        # looks like [0,1]
        Xf = Xf * 255.0
        print("Detected samples in [0,1]; converted to raw [0,255].")
    else:
        print("Detected samples in raw-like scale; using as [0,255].")

    return Xf, y.long()


class CuckooSearchSingleSample:
    def __init__(self,
                 n_nests: int,
                 n_iterations: int,
                 dim: int,
                 model: nn.Module,
                 X_sample_chw: torch.Tensor,   # [3,32,32] raw
                 y_sample: torch.Tensor,       # scalar or [1]
                 p_a: float = 0.5,
                 lambda_param: float = 1e8,
                 step_size: float = 2.0,
                 step_decay: float = 0.98):
        self.n_nests = n_nests
        self.n_iterations = n_iterations
        self.dim = dim
        self.model = model
        self.X = X_sample_chw.unsqueeze(0).to(device)  # [1,3,32,32]
        self.y = y_sample.to(device)
        self.p_a = p_a
        self.lambda_param = lambda_param
        self.step_size = step_size
        self.step_decay = step_decay

        self.nests = torch.zeros((n_nests, dim), dtype=torch.float32, device=device)
        self.fitness = torch.empty(n_nests, dtype=torch.float32, device=device)

        for i in range(n_nests):
            self.fitness[i] = self._fitness(self.nests[i])

        best_idx = torch.argmin(self.fitness)
        self.best_nest = self.nests[best_idx].clone()
        self.best_fitness = float(self.fitness[best_idx].item())

    def _cauchy_flight(self) -> torch.Tensor:
        dist = torch.distributions.Cauchy(loc=torch.tensor(0.0, device=device),
                                          scale=torch.tensor(self.step_size, device=device))
        return dist.sample((self.dim,))

    def _new_solution(self, curr: torch.Tensor) -> torch.Tensor:
        cand = curr + self._cauchy_flight()
        return clamp_tensor(cand, raw_lower, raw_upper)

    def _fitness(self, perturb_flat: torch.Tensor) -> float:
        p = clamp_tensor(perturb_flat, raw_lower, raw_upper).view(1, *img_shape)  # [1,3,32,32]
        x_adv_raw = self.X + p
        loss = calculate_adversarial_loss(self.model, x_adv_raw, self.y).item()
        mag = torch.norm(p).item()
        return mag - self.lambda_param * loss

    def optimize(self) -> None:
        for it in range(1, self.n_iterations + 1):
            for i in range(self.n_nests):
                cand = self._new_solution(self.nests[i])
                cand_fit = self._fitness(cand)
                if cand_fit < self.fitness[i]:
                    self.nests[i] = cand
                    self.fitness[i] = cand_fit
                    if cand_fit < self.best_fitness:
                        self.best_fitness = cand_fit
                        self.best_nest = cand.clone()

            k = int(self.n_nests * self.p_a)
            if k > 0:
                worst_idx = torch.argsort(self.fitness)[-k:]
                for idx in worst_idx:
                    new_rand = torch.empty((self.dim,), device=device).uniform_(float(raw_lower), float(raw_upper))
                    self.nests[idx] = new_rand
                    self.fitness[idx] = self._fitness(new_rand)

            if it in {1, self.n_iterations} or it % 10 == 0:
                print(f"    iter {it}/{self.n_iterations} — best fitness {self.best_fitness:.5f}")

            self.step_size *= self.step_decay

    def get_best_solution(self) -> torch.Tensor:
        return self.best_nest


def main():
    MODEL_PATH = "/content/drive/MyDrive/ZC-CSA/Resnet.pt"
    SAMPLES_PATH = "/content/drive/MyDrive/ZC-CSA/selected_samples_for_attack_cifar10.pt"

    model = load_model(MODEL_PATH)
    X_full, y_full = load_samples(SAMPLES_PATH)

    # outputs
    base_dir = "/content/drive/MyDrive/including_cifar10"
    orig_dir = os.path.join(base_dir, "Original")
    adv_dir  = os.path.join(base_dir, "Adversarial")
    pert_dir = os.path.join(base_dir, "Perturbations")
    for d in (orig_dir, adv_dir, pert_dir):
        os.makedirs(d, exist_ok=True)

    # hyperparams (you will likely tune lambda_param / step_size / bounds for CIFAR)
    n_nests = 58
    n_iterations = 100
    p_a = 0.5
    lambda_param = 1e8
    step_size = 2.0
    step_decay = 0.98

    results = []
    processed = 0
    success_count = 0

    X_full = X_full.to(device)
    y_full = y_full.to(device)

    tic = time.time()
    for cls in range(4,6):
        idxs = (y_full == cls).nonzero(as_tuple=True)[0]
        if idxs.numel() == 0:
            print(f"No samples for class {cls}; skipping.")
            continue

        print(f"\n── Class {cls}: {idxs.numel()} samples ──")
        for j, gidx in enumerate(idxs):
            x = X_full[gidx]              # [3,32,32] raw
            y = y_full[gidx]              # scalar
            gidx_int = int(gidx.item())

            print(f"  sample {j+1}/{idxs.numel()} (global {gidx_int})")

            save_cifar_uint8_rgb(x.detach().cpu().numpy(),
                                 os.path.join(orig_dir, f"{gidx_int}_lbl{cls}.png"))

            css = CuckooSearchSingleSample(
                n_nests=n_nests,
                n_iterations=n_iterations,
                dim=dim,
                model=model,
                X_sample_chw=x,
                y_sample=y,
                p_a=p_a,
                lambda_param=lambda_param,
                step_size=step_size,
                step_decay=step_decay
            )
            css.optimize()
            delta_flat = css.get_best_solution()
            delta = delta_flat.view(*img_shape)

            x_adv_raw = clamp_tensor(x + delta, final_lower, final_upper)

            with torch.no_grad():
                pred = torch.argmax(model(to_model_input(x_adv_raw.unsqueeze(0))), dim=1).item()

            l2 = float(torch.norm(delta).item())
            miscls = int(pred != int(y.item()))

            # save perturbation
            delta_np = delta.detach().cpu().numpy()
            np.save(os.path.join(pert_dir, f"{gidx_int}_delta.npy"), delta_np)

            # visualize delta (map [-B,+B] -> [0,255])
            B = max(abs(raw_lower_bound), abs(raw_upper_bound))
            delta_vis = (np.clip(delta_np, -B, B) + B) / (2 * B) * 255.0
            save_cifar_uint8_rgb(delta_vis, os.path.join(pert_dir, f"{gidx_int}_delta_vis.png"))

            if miscls:
                success_count += 1
                save_cifar_uint8_rgb(x_adv_raw.detach().cpu().numpy(),
                                     os.path.join(adv_dir, f"{gidx_int}_t{cls}_p{pred}_m{l2:.2f}.png"))

            results.append({
                "class": cls,
                "index": gidx_int,
                "true": int(y.item()),
                "pred": int(pred),
                "misclassified": bool(miscls),
                "perturb_norm_l2": l2
            })
            processed += 1

    df = pd.DataFrame(results)
    os.makedirs(base_dir, exist_ok=True)
    csv_path = os.path.join(base_dir, "4_5_zc_csa_cifar10_results.csv")
    df.to_csv(csv_path, index=False)

    stats = (df.groupby(["class", "misclassified"])
               .size()
               .unstack(fill_value=0)
               .rename(columns={False: "correct", True: "misclassified"}))
    if "misclassified" not in stats.columns:
        stats["misclassified"] = 0
    if "correct" not in stats.columns:
        stats["correct"] = 0
    stats["total"] = stats["correct"] + stats["misclassified"]
    stats["correct_%"] = (stats["correct"]/stats["total"]*100).round(2)
    stats["miscls_%"]  = (stats["misclassified"]/stats["total"]*100).round(2)

    stats_path = os.path.join(base_dir, "4_5_zc_csa_cifar10_stats_per_class.csv")
    stats.to_csv(stats_path)

    toc = time.time()
    print("\n===== SUMMARY =====")
    print(f"Processed samples        : {processed}")
    print(f"Successful attacks       : {success_count}")
    print(f"Results CSV              : {csv_path}")
    print(f"Per-class stats CSV      : {stats_path}")
    print(f"Total time               : {toc - tic:.2f} s\n")
    print(stats.to_string())


if __name__ == "__main__":
    main()

Loaded TorchScript model from /content/drive/MyDrive/ZC-CSA/Resnet.pt
Loaded object type: <class 'dict'>
Loaded dictionary keys: dict_keys(['images', 'labels'])
  Key 'images': Tensor shape torch.Size([1000, 3, 32, 32]), dtype torch.float32
  Key 'labels': Tensor shape torch.Size([1000]), dtype torch.int64
Detected samples in [0,1]; converted to raw [0,255].

── Class 4: 100 samples ──
  sample 1/100 (global 400)
    iter 1/100 — best fitness -246632768.00000
    iter 10/100 — best fitness -246632768.00000
    iter 20/100 — best fitness -246632768.00000
    iter 30/100 — best fitness -246632768.00000
    iter 40/100 — best fitness -246632768.00000
    iter 50/100 — best fitness -246632768.00000
    iter 60/100 — best fitness -246632768.00000
    iter 70/100 — best fitness -246632768.00000
    iter 80/100 — best fitness -246632768.00000
    iter 90/100 — best fitness -246798563.81038
    iter 100/100 — best fitness -247153839.50861
  sample 2/100 (global 401)
    iter 1/100 — best fitne

# On Classes 6,7

In [3]:
import os
import time
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd

# -------------------------
# CIFAR10 setup
# -------------------------
img_shape = (3, 32, 32)
dim = int(np.prod(img_shape))

# Pixel bounds in RAW space:
# We'll assume samples are uint8-like in [0,255] unless detected otherwise.
final_lower_bound = 0.0
final_upper_bound = 255.0

# Perturbation bounds (tune as you like for CIFAR):
# These are per-pixel deltas in raw [0..255] space.
raw_lower_bound = -20.0
raw_upper_bound = 20.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

raw_lower   = torch.tensor(raw_lower_bound,   dtype=torch.float32, device=device)
raw_upper   = torch.tensor(raw_upper_bound,   dtype=torch.float32, device=device)
final_lower = torch.tensor(final_lower_bound, dtype=torch.float32, device=device)
final_upper = torch.tensor(final_upper_bound, dtype=torch.float32, device=device)

# CIFAR10 normalization (match your training!)
CIFAR10_MEAN = torch.tensor([0.4914, 0.4822, 0.4465], device=device).view(1,3,1,1)
CIFAR10_STD  = torch.tensor([0.2470, 0.2435, 0.2616], device=device).view(1,3,1,1)

# Always CNN mode for ResNet
CNN_MODE = True


def clamp_tensor(x: torch.Tensor, lower: torch.Tensor, upper: torch.Tensor) -> torch.Tensor:
    return torch.max(torch.min(x, upper), lower)


def to_model_input(x_raw: torch.Tensor) -> torch.Tensor:
    """
    x_raw: [1,3,32,32] in raw pixel space.
    Returns normalized tensor suitable for ResNet.
    """
    x_clip = clamp_tensor(x_raw, final_lower, final_upper)
    x_01 = x_clip / final_upper  # [0,1]
    x_norm = (x_01 - CIFAR10_MEAN) / CIFAR10_STD
    return x_norm


def calculate_adversarial_loss(model: nn.Module, x_raw_single: torch.Tensor, y_single: torch.Tensor, eps: float = 1e-9) -> torch.Tensor:
    """
    x_raw_single: [1,3,32,32] raw pixels
    y_single: scalar tensor or shape [1]
    """
    if y_single.ndim == 0:
        y_single = y_single.view(1)

    x_in = to_model_input(x_raw_single)
    logits = model(x_in)

    log_probs = F.log_softmax(logits, dim=1)
    ce = -log_probs[0, y_single.item()]
    return torch.clamp(ce, min=eps)


def save_cifar_uint8_rgb(img_chw: np.ndarray, path: str) -> None:
    """
    img_chw: (3,32,32) float or uint8 in [0,255]
    Saves as PNG. Converts RGB->BGR for OpenCV.
    """
    img = np.clip(np.round(img_chw), 0, 255).astype(np.uint8)
    img_hwc = np.transpose(img, (1,2,0))          # HWC RGB
    img_bgr = img_hwc[..., ::-1]                  # RGB->BGR
    cv2.imwrite(path, img_bgr)


def load_model(model_path: str) -> nn.Module:
    # Try TorchScript first
    try:
        m = torch.jit.load(model_path, map_location=device)
        m.eval().to(device)
        print(f"Loaded TorchScript model from {model_path}")
        return m
    except Exception as e1:
        # Fallback: regular torch.load (could be full model or state_dict)
        ckpt = torch.load(model_path, map_location=device)
        if isinstance(ckpt, nn.Module):
            m = ckpt
            m.eval().to(device)
            print(f"Loaded nn.Module from torch.load: {model_path}")
            return m
        raise RuntimeError(
            f"Could not load model as TorchScript or nn.Module. "
            f"If this is a state_dict, you must rebuild ResNet18 and load_state_dict.\n"
            f"TorchScript error: {e1}"
        )


def load_samples(samples_path: str):
    obj = torch.load(samples_path, map_location="cpu")

    # Debugging: Print type and content of obj
    print(f"Loaded object type: {type(obj)}")
    if isinstance(obj, dict):
        print(f"Loaded dictionary keys: {obj.keys()}")
        for k, v in obj.items():
            if torch.is_tensor(v):
                print(f"  Key '{k}': Tensor shape {v.shape}, dtype {v.dtype}")
            else:
                print(f"  Key '{k}': Type {type(v)}")
    elif isinstance(obj, (tuple, list)):
        print(f"Loaded tuple/list length: {len(obj)}")
        for i, item in enumerate(obj):
            if torch.is_tensor(item):
                print(f"  Item {i}: Tensor shape {item.shape}, dtype {item.dtype}")
            else:
                print(f"  Item {i}: Type {type(item)}")
    else:
        print(f"Loaded object value (first 100 chars): {str(obj)[:100]}")


    # Common patterns:
    if isinstance(obj, (tuple, list)) and len(obj) == 2 and torch.is_tensor(obj[0]) and torch.is_tensor(obj[1]):
        X, y = obj
    elif isinstance(obj, dict) and ("images" in obj and "labels" in obj):
        X, y = obj["images"], obj["labels"]
    elif isinstance(obj, dict) and ("X" in obj and "y" in obj):
        X, y = obj["X"], obj["y"]
    else:
        raise ValueError("Unrecognized samples format. Expected (X,y) tuple or {'X':..., 'y':...} dict.")

    # Ensure shapes: X [N,3,32,32], y [N]
    if X.ndim == 2 and X.shape[1] == dim:
        X = X.view(-1, *img_shape)
    if X.ndim != 4:
        raise ValueError(f"X must be [N,3,32,32]. Got {X.shape}")
    if y.ndim != 1:
        y = y.view(-1)

    # Detect whether X is already 0..255 or 0..1
    Xf = X.float()
    maxv = float(Xf.max().item())
    if maxv <= 1.5:
        # looks like [0,1]
        Xf = Xf * 255.0
        print("Detected samples in [0,1]; converted to raw [0,255].")
    else:
        print("Detected samples in raw-like scale; using as [0,255].")

    return Xf, y.long()


class CuckooSearchSingleSample:
    def __init__(self,
                 n_nests: int,
                 n_iterations: int,
                 dim: int,
                 model: nn.Module,
                 X_sample_chw: torch.Tensor,   # [3,32,32] raw
                 y_sample: torch.Tensor,       # scalar or [1]
                 p_a: float = 0.5,
                 lambda_param: float = 1e8,
                 step_size: float = 2.0,
                 step_decay: float = 0.98):
        self.n_nests = n_nests
        self.n_iterations = n_iterations
        self.dim = dim
        self.model = model
        self.X = X_sample_chw.unsqueeze(0).to(device)  # [1,3,32,32]
        self.y = y_sample.to(device)
        self.p_a = p_a
        self.lambda_param = lambda_param
        self.step_size = step_size
        self.step_decay = step_decay

        self.nests = torch.zeros((n_nests, dim), dtype=torch.float32, device=device)
        self.fitness = torch.empty(n_nests, dtype=torch.float32, device=device)

        for i in range(n_nests):
            self.fitness[i] = self._fitness(self.nests[i])

        best_idx = torch.argmin(self.fitness)
        self.best_nest = self.nests[best_idx].clone()
        self.best_fitness = float(self.fitness[best_idx].item())

    def _cauchy_flight(self) -> torch.Tensor:
        dist = torch.distributions.Cauchy(loc=torch.tensor(0.0, device=device),
                                          scale=torch.tensor(self.step_size, device=device))
        return dist.sample((self.dim,))

    def _new_solution(self, curr: torch.Tensor) -> torch.Tensor:
        cand = curr + self._cauchy_flight()
        return clamp_tensor(cand, raw_lower, raw_upper)

    def _fitness(self, perturb_flat: torch.Tensor) -> float:
        p = clamp_tensor(perturb_flat, raw_lower, raw_upper).view(1, *img_shape)  # [1,3,32,32]
        x_adv_raw = self.X + p
        loss = calculate_adversarial_loss(self.model, x_adv_raw, self.y).item()
        mag = torch.norm(p).item()
        return mag - self.lambda_param * loss

    def optimize(self) -> None:
        for it in range(1, self.n_iterations + 1):
            for i in range(self.n_nests):
                cand = self._new_solution(self.nests[i])
                cand_fit = self._fitness(cand)
                if cand_fit < self.fitness[i]:
                    self.nests[i] = cand
                    self.fitness[i] = cand_fit
                    if cand_fit < self.best_fitness:
                        self.best_fitness = cand_fit
                        self.best_nest = cand.clone()

            k = int(self.n_nests * self.p_a)
            if k > 0:
                worst_idx = torch.argsort(self.fitness)[-k:]
                for idx in worst_idx:
                    new_rand = torch.empty((self.dim,), device=device).uniform_(float(raw_lower), float(raw_upper))
                    self.nests[idx] = new_rand
                    self.fitness[idx] = self._fitness(new_rand)

            if it in {1, self.n_iterations} or it % 10 == 0:
                print(f"    iter {it}/{self.n_iterations} — best fitness {self.best_fitness:.5f}")

            self.step_size *= self.step_decay

    def get_best_solution(self) -> torch.Tensor:
        return self.best_nest


def main():
    MODEL_PATH = "/content/drive/MyDrive/ZC-CSA/Resnet.pt"
    SAMPLES_PATH = "/content/drive/MyDrive/ZC-CSA/selected_samples_for_attack_cifar10.pt"

    model = load_model(MODEL_PATH)
    X_full, y_full = load_samples(SAMPLES_PATH)

    # outputs
    base_dir = "/content/drive/MyDrive/including_cifar10"
    orig_dir = os.path.join(base_dir, "Original")
    adv_dir  = os.path.join(base_dir, "Adversarial")
    pert_dir = os.path.join(base_dir, "Perturbations")
    for d in (orig_dir, adv_dir, pert_dir):
        os.makedirs(d, exist_ok=True)

    # hyperparams (you will likely tune lambda_param / step_size / bounds for CIFAR)
    n_nests = 58
    n_iterations = 100
    p_a = 0.5
    lambda_param = 1e8
    step_size = 2.0
    step_decay = 0.98

    results = []
    processed = 0
    success_count = 0

    X_full = X_full.to(device)
    y_full = y_full.to(device)

    tic = time.time()
    for cls in range(6,8):
        idxs = (y_full == cls).nonzero(as_tuple=True)[0]
        if idxs.numel() == 0:
            print(f"No samples for class {cls}; skipping.")
            continue

        print(f"\n── Class {cls}: {idxs.numel()} samples ──")
        for j, gidx in enumerate(idxs):
            x = X_full[gidx]              # [3,32,32] raw
            y = y_full[gidx]              # scalar
            gidx_int = int(gidx.item())

            print(f"  sample {j+1}/{idxs.numel()} (global {gidx_int})")

            save_cifar_uint8_rgb(x.detach().cpu().numpy(),
                                 os.path.join(orig_dir, f"{gidx_int}_lbl{cls}.png"))

            css = CuckooSearchSingleSample(
                n_nests=n_nests,
                n_iterations=n_iterations,
                dim=dim,
                model=model,
                X_sample_chw=x,
                y_sample=y,
                p_a=p_a,
                lambda_param=lambda_param,
                step_size=step_size,
                step_decay=step_decay
            )
            css.optimize()
            delta_flat = css.get_best_solution()
            delta = delta_flat.view(*img_shape)

            x_adv_raw = clamp_tensor(x + delta, final_lower, final_upper)

            with torch.no_grad():
                pred = torch.argmax(model(to_model_input(x_adv_raw.unsqueeze(0))), dim=1).item()

            l2 = float(torch.norm(delta).item())
            miscls = int(pred != int(y.item()))

            # save perturbation
            delta_np = delta.detach().cpu().numpy()
            np.save(os.path.join(pert_dir, f"{gidx_int}_delta.npy"), delta_np)

            # visualize delta (map [-B,+B] -> [0,255])
            B = max(abs(raw_lower_bound), abs(raw_upper_bound))
            delta_vis = (np.clip(delta_np, -B, B) + B) / (2 * B) * 255.0
            save_cifar_uint8_rgb(delta_vis, os.path.join(pert_dir, f"{gidx_int}_delta_vis.png"))

            if miscls:
                success_count += 1
                save_cifar_uint8_rgb(x_adv_raw.detach().cpu().numpy(),
                                     os.path.join(adv_dir, f"{gidx_int}_t{cls}_p{pred}_m{l2:.2f}.png"))

            results.append({
                "class": cls,
                "index": gidx_int,
                "true": int(y.item()),
                "pred": int(pred),
                "misclassified": bool(miscls),
                "perturb_norm_l2": l2
            })
            processed += 1

    df = pd.DataFrame(results)
    os.makedirs(base_dir, exist_ok=True)
    csv_path = os.path.join(base_dir, "6_7_zc_csa_cifar10_results.csv")
    df.to_csv(csv_path, index=False)

    stats = (df.groupby(["class", "misclassified"])
               .size()
               .unstack(fill_value=0)
               .rename(columns={False: "correct", True: "misclassified"}))
    if "misclassified" not in stats.columns:
        stats["misclassified"] = 0
    if "correct" not in stats.columns:
        stats["correct"] = 0
    stats["total"] = stats["correct"] + stats["misclassified"]
    stats["correct_%"] = (stats["correct"]/stats["total"]*100).round(2)
    stats["miscls_%"]  = (stats["misclassified"]/stats["total"]*100).round(2)

    stats_path = os.path.join(base_dir, "6_7_zc_csa_cifar10_stats_per_class.csv")
    stats.to_csv(stats_path)

    toc = time.time()
    print("\n===== SUMMARY =====")
    print(f"Processed samples        : {processed}")
    print(f"Successful attacks       : {success_count}")
    print(f"Results CSV              : {csv_path}")
    print(f"Per-class stats CSV      : {stats_path}")
    print(f"Total time               : {toc - tic:.2f} s\n")
    print(stats.to_string())


if __name__ == "__main__":
    main()

Loaded TorchScript model from /content/drive/MyDrive/ZC-CSA/Resnet.pt
Loaded object type: <class 'dict'>
Loaded dictionary keys: dict_keys(['images', 'labels'])
  Key 'images': Tensor shape torch.Size([1000, 3, 32, 32]), dtype torch.float32
  Key 'labels': Tensor shape torch.Size([1000]), dtype torch.int64
Detected samples in [0,1]; converted to raw [0,255].

── Class 6: 100 samples ──
  sample 1/100 (global 600)
    iter 1/100 — best fitness -147440978.44727
    iter 10/100 — best fitness -233960114.00732
    iter 20/100 — best fitness -254920078.41498
    iter 30/100 — best fitness -260863819.21185
    iter 40/100 — best fitness -286219211.55530
    iter 50/100 — best fitness -303203459.65906
    iter 60/100 — best fitness -303492759.35834
    iter 70/100 — best fitness -347782551.84302
    iter 80/100 — best fitness -349588334.85529
    iter 90/100 — best fitness -355255351.10541
    iter 100/100 — best fitness -364638720.13177
  sample 2/100 (global 601)
    iter 1/100 — best fitne

# On Classess 8,9

In [4]:
import os
import time
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd

# -------------------------
# CIFAR10 setup
# -------------------------
img_shape = (3, 32, 32)
dim = int(np.prod(img_shape))

# Pixel bounds in RAW space:
# We'll assume samples are uint8-like in [0,255] unless detected otherwise.
final_lower_bound = 0.0
final_upper_bound = 255.0

# Perturbation bounds (tune as you like for CIFAR):
# These are per-pixel deltas in raw [0..255] space.
raw_lower_bound = -20.0
raw_upper_bound = 20.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

raw_lower   = torch.tensor(raw_lower_bound,   dtype=torch.float32, device=device)
raw_upper   = torch.tensor(raw_upper_bound,   dtype=torch.float32, device=device)
final_lower = torch.tensor(final_lower_bound, dtype=torch.float32, device=device)
final_upper = torch.tensor(final_upper_bound, dtype=torch.float32, device=device)

# CIFAR10 normalization (match your training!)
CIFAR10_MEAN = torch.tensor([0.4914, 0.4822, 0.4465], device=device).view(1,3,1,1)
CIFAR10_STD  = torch.tensor([0.2470, 0.2435, 0.2616], device=device).view(1,3,1,1)

# Always CNN mode for ResNet
CNN_MODE = True


def clamp_tensor(x: torch.Tensor, lower: torch.Tensor, upper: torch.Tensor) -> torch.Tensor:
    return torch.max(torch.min(x, upper), lower)


def to_model_input(x_raw: torch.Tensor) -> torch.Tensor:
    """
    x_raw: [1,3,32,32] in raw pixel space.
    Returns normalized tensor suitable for ResNet.
    """
    x_clip = clamp_tensor(x_raw, final_lower, final_upper)
    x_01 = x_clip / final_upper  # [0,1]
    x_norm = (x_01 - CIFAR10_MEAN) / CIFAR10_STD
    return x_norm


def calculate_adversarial_loss(model: nn.Module, x_raw_single: torch.Tensor, y_single: torch.Tensor, eps: float = 1e-9) -> torch.Tensor:
    """
    x_raw_single: [1,3,32,32] raw pixels
    y_single: scalar tensor or shape [1]
    """
    if y_single.ndim == 0:
        y_single = y_single.view(1)

    x_in = to_model_input(x_raw_single)
    logits = model(x_in)

    log_probs = F.log_softmax(logits, dim=1)
    ce = -log_probs[0, y_single.item()]
    return torch.clamp(ce, min=eps)


def save_cifar_uint8_rgb(img_chw: np.ndarray, path: str) -> None:
    """
    img_chw: (3,32,32) float or uint8 in [0,255]
    Saves as PNG. Converts RGB->BGR for OpenCV.
    """
    img = np.clip(np.round(img_chw), 0, 255).astype(np.uint8)
    img_hwc = np.transpose(img, (1,2,0))          # HWC RGB
    img_bgr = img_hwc[..., ::-1]                  # RGB->BGR
    cv2.imwrite(path, img_bgr)


def load_model(model_path: str) -> nn.Module:
    # Try TorchScript first
    try:
        m = torch.jit.load(model_path, map_location=device)
        m.eval().to(device)
        print(f"Loaded TorchScript model from {model_path}")
        return m
    except Exception as e1:
        # Fallback: regular torch.load (could be full model or state_dict)
        ckpt = torch.load(model_path, map_location=device)
        if isinstance(ckpt, nn.Module):
            m = ckpt
            m.eval().to(device)
            print(f"Loaded nn.Module from torch.load: {model_path}")
            return m
        raise RuntimeError(
            f"Could not load model as TorchScript or nn.Module. "
            f"If this is a state_dict, you must rebuild ResNet18 and load_state_dict.\n"
            f"TorchScript error: {e1}"
        )


def load_samples(samples_path: str):
    obj = torch.load(samples_path, map_location="cpu")

    # Debugging: Print type and content of obj
    print(f"Loaded object type: {type(obj)}")
    if isinstance(obj, dict):
        print(f"Loaded dictionary keys: {obj.keys()}")
        for k, v in obj.items():
            if torch.is_tensor(v):
                print(f"  Key '{k}': Tensor shape {v.shape}, dtype {v.dtype}")
            else:
                print(f"  Key '{k}': Type {type(v)}")
    elif isinstance(obj, (tuple, list)):
        print(f"Loaded tuple/list length: {len(obj)}")
        for i, item in enumerate(obj):
            if torch.is_tensor(item):
                print(f"  Item {i}: Tensor shape {item.shape}, dtype {item.dtype}")
            else:
                print(f"  Item {i}: Type {type(item)}")
    else:
        print(f"Loaded object value (first 100 chars): {str(obj)[:100]}")


    # Common patterns:
    if isinstance(obj, (tuple, list)) and len(obj) == 2 and torch.is_tensor(obj[0]) and torch.is_tensor(obj[1]):
        X, y = obj
    elif isinstance(obj, dict) and ("images" in obj and "labels" in obj):
        X, y = obj["images"], obj["labels"]
    elif isinstance(obj, dict) and ("X" in obj and "y" in obj):
        X, y = obj["X"], obj["y"]
    else:
        raise ValueError("Unrecognized samples format. Expected (X,y) tuple or {'X':..., 'y':...} dict.")

    # Ensure shapes: X [N,3,32,32], y [N]
    if X.ndim == 2 and X.shape[1] == dim:
        X = X.view(-1, *img_shape)
    if X.ndim != 4:
        raise ValueError(f"X must be [N,3,32,32]. Got {X.shape}")
    if y.ndim != 1:
        y = y.view(-1)

    # Detect whether X is already 0..255 or 0..1
    Xf = X.float()
    maxv = float(Xf.max().item())
    if maxv <= 1.5:
        # looks like [0,1]
        Xf = Xf * 255.0
        print("Detected samples in [0,1]; converted to raw [0,255].")
    else:
        print("Detected samples in raw-like scale; using as [0,255].")

    return Xf, y.long()


class CuckooSearchSingleSample:
    def __init__(self,
                 n_nests: int,
                 n_iterations: int,
                 dim: int,
                 model: nn.Module,
                 X_sample_chw: torch.Tensor,   # [3,32,32] raw
                 y_sample: torch.Tensor,       # scalar or [1]
                 p_a: float = 0.5,
                 lambda_param: float = 1e8,
                 step_size: float = 2.0,
                 step_decay: float = 0.98):
        self.n_nests = n_nests
        self.n_iterations = n_iterations
        self.dim = dim
        self.model = model
        self.X = X_sample_chw.unsqueeze(0).to(device)  # [1,3,32,32]
        self.y = y_sample.to(device)
        self.p_a = p_a
        self.lambda_param = lambda_param
        self.step_size = step_size
        self.step_decay = step_decay

        self.nests = torch.zeros((n_nests, dim), dtype=torch.float32, device=device)
        self.fitness = torch.empty(n_nests, dtype=torch.float32, device=device)

        for i in range(n_nests):
            self.fitness[i] = self._fitness(self.nests[i])

        best_idx = torch.argmin(self.fitness)
        self.best_nest = self.nests[best_idx].clone()
        self.best_fitness = float(self.fitness[best_idx].item())

    def _cauchy_flight(self) -> torch.Tensor:
        dist = torch.distributions.Cauchy(loc=torch.tensor(0.0, device=device),
                                          scale=torch.tensor(self.step_size, device=device))
        return dist.sample((self.dim,))

    def _new_solution(self, curr: torch.Tensor) -> torch.Tensor:
        cand = curr + self._cauchy_flight()
        return clamp_tensor(cand, raw_lower, raw_upper)

    def _fitness(self, perturb_flat: torch.Tensor) -> float:
        p = clamp_tensor(perturb_flat, raw_lower, raw_upper).view(1, *img_shape)  # [1,3,32,32]
        x_adv_raw = self.X + p
        loss = calculate_adversarial_loss(self.model, x_adv_raw, self.y).item()
        mag = torch.norm(p).item()
        return mag - self.lambda_param * loss

    def optimize(self) -> None:
        for it in range(1, self.n_iterations + 1):
            for i in range(self.n_nests):
                cand = self._new_solution(self.nests[i])
                cand_fit = self._fitness(cand)
                if cand_fit < self.fitness[i]:
                    self.nests[i] = cand
                    self.fitness[i] = cand_fit
                    if cand_fit < self.best_fitness:
                        self.best_fitness = cand_fit
                        self.best_nest = cand.clone()

            k = int(self.n_nests * self.p_a)
            if k > 0:
                worst_idx = torch.argsort(self.fitness)[-k:]
                for idx in worst_idx:
                    new_rand = torch.empty((self.dim,), device=device).uniform_(float(raw_lower), float(raw_upper))
                    self.nests[idx] = new_rand
                    self.fitness[idx] = self._fitness(new_rand)

            if it in {1, self.n_iterations} or it % 10 == 0:
                print(f"    iter {it}/{self.n_iterations} — best fitness {self.best_fitness:.5f}")

            self.step_size *= self.step_decay

    def get_best_solution(self) -> torch.Tensor:
        return self.best_nest


def main():
    MODEL_PATH = "/content/drive/MyDrive/ZC-CSA/Resnet.pt"
    SAMPLES_PATH = "/content/drive/MyDrive/ZC-CSA/selected_samples_for_attack_cifar10.pt"

    model = load_model(MODEL_PATH)
    X_full, y_full = load_samples(SAMPLES_PATH)

    # outputs
    base_dir = "/content/drive/MyDrive/including_cifar10"
    orig_dir = os.path.join(base_dir, "Original")
    adv_dir  = os.path.join(base_dir, "Adversarial")
    pert_dir = os.path.join(base_dir, "Perturbations")
    for d in (orig_dir, adv_dir, pert_dir):
        os.makedirs(d, exist_ok=True)

    # hyperparams (you will likely tune lambda_param / step_size / bounds for CIFAR)
    n_nests = 58
    n_iterations = 100
    p_a = 0.5
    lambda_param = 1e8
    step_size = 2.0
    step_decay = 0.98

    results = []
    processed = 0
    success_count = 0

    X_full = X_full.to(device)
    y_full = y_full.to(device)

    tic = time.time()
    for cls in range(8,10):
        idxs = (y_full == cls).nonzero(as_tuple=True)[0]
        if idxs.numel() == 0:
            print(f"No samples for class {cls}; skipping.")
            continue

        print(f"\n── Class {cls}: {idxs.numel()} samples ──")
        for j, gidx in enumerate(idxs):
            x = X_full[gidx]              # [3,32,32] raw
            y = y_full[gidx]              # scalar
            gidx_int = int(gidx.item())

            print(f"  sample {j+1}/{idxs.numel()} (global {gidx_int})")

            save_cifar_uint8_rgb(x.detach().cpu().numpy(),
                                 os.path.join(orig_dir, f"{gidx_int}_lbl{cls}.png"))

            css = CuckooSearchSingleSample(
                n_nests=n_nests,
                n_iterations=n_iterations,
                dim=dim,
                model=model,
                X_sample_chw=x,
                y_sample=y,
                p_a=p_a,
                lambda_param=lambda_param,
                step_size=step_size,
                step_decay=step_decay
            )
            css.optimize()
            delta_flat = css.get_best_solution()
            delta = delta_flat.view(*img_shape)

            x_adv_raw = clamp_tensor(x + delta, final_lower, final_upper)

            with torch.no_grad():
                pred = torch.argmax(model(to_model_input(x_adv_raw.unsqueeze(0))), dim=1).item()

            l2 = float(torch.norm(delta).item())
            miscls = int(pred != int(y.item()))

            # save perturbation
            delta_np = delta.detach().cpu().numpy()
            np.save(os.path.join(pert_dir, f"{gidx_int}_delta.npy"), delta_np)

            # visualize delta (map [-B,+B] -> [0,255])
            B = max(abs(raw_lower_bound), abs(raw_upper_bound))
            delta_vis = (np.clip(delta_np, -B, B) + B) / (2 * B) * 255.0
            save_cifar_uint8_rgb(delta_vis, os.path.join(pert_dir, f"{gidx_int}_delta_vis.png"))

            if miscls:
                success_count += 1
                save_cifar_uint8_rgb(x_adv_raw.detach().cpu().numpy(),
                                     os.path.join(adv_dir, f"{gidx_int}_t{cls}_p{pred}_m{l2:.2f}.png"))

            results.append({
                "class": cls,
                "index": gidx_int,
                "true": int(y.item()),
                "pred": int(pred),
                "misclassified": bool(miscls),
                "perturb_norm_l2": l2
            })
            processed += 1

    df = pd.DataFrame(results)
    os.makedirs(base_dir, exist_ok=True)
    csv_path = os.path.join(base_dir, "8_9_zc_csa_cifar10_results.csv")
    df.to_csv(csv_path, index=False)

    stats = (df.groupby(["class", "misclassified"])
               .size()
               .unstack(fill_value=0)
               .rename(columns={False: "correct", True: "misclassified"}))
    if "misclassified" not in stats.columns:
        stats["misclassified"] = 0
    if "correct" not in stats.columns:
        stats["correct"] = 0
    stats["total"] = stats["correct"] + stats["misclassified"]
    stats["correct_%"] = (stats["correct"]/stats["total"]*100).round(2)
    stats["miscls_%"]  = (stats["misclassified"]/stats["total"]*100).round(2)

    stats_path = os.path.join(base_dir, "8_9_zc_csa_cifar10_stats_per_class.csv")
    stats.to_csv(stats_path)

    toc = time.time()
    print("\n===== SUMMARY =====")
    print(f"Processed samples        : {processed}")
    print(f"Successful attacks       : {success_count}")
    print(f"Results CSV              : {csv_path}")
    print(f"Per-class stats CSV      : {stats_path}")
    print(f"Total time               : {toc - tic:.2f} s\n")
    print(stats.to_string())


if __name__ == "__main__":
    main()

Loaded TorchScript model from /content/drive/MyDrive/ZC-CSA/Resnet.pt
Loaded object type: <class 'dict'>
Loaded dictionary keys: dict_keys(['images', 'labels'])
  Key 'images': Tensor shape torch.Size([1000, 3, 32, 32]), dtype torch.float32
  Key 'labels': Tensor shape torch.Size([1000]), dtype torch.int64
Detected samples in [0,1]; converted to raw [0,255].

── Class 8: 100 samples ──
  sample 1/100 (global 800)
    iter 1/100 — best fitness -300288034.62820
    iter 10/100 — best fitness -440176806.54816
    iter 20/100 — best fitness -445384238.42902
    iter 30/100 — best fitness -480268546.50873
    iter 40/100 — best fitness -506937290.07922
    iter 50/100 — best fitness -506937290.07922
    iter 60/100 — best fitness -518208286.85809
    iter 70/100 — best fitness -526698991.69092
    iter 80/100 — best fitness -529418682.76562
    iter 90/100 — best fitness -529724768.39716
    iter 100/100 — best fitness -533656053.10406
  sample 2/100 (global 801)
    iter 1/100 — best fitne

# **Following is for MNIST**

In [ ]:
'''
import os
import time
import joblib
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# ──────────────────────────────────────────────────────────
# Numeric bounds & device
# ──────────────────────────────────────────────────────────
raw_upper_bound  = 50.0
raw_lower_bound  = -70.0
final_lower_bound = 0.0
final_upper_bound = 255.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

raw_lower  = torch.tensor(raw_lower_bound,  dtype=torch.float32, device=device)
raw_upper  = torch.tensor(raw_upper_bound,  dtype=torch.float32, device=device)
final_lower = torch.tensor(final_lower_bound, dtype=torch.float32, device=device)
final_upper = torch.tensor(final_upper_bound, dtype=torch.float32, device=device)

img_shape = (28, 28)
dim = img_shape[0] * img_shape[1]

# ──────────────────────────────────────────────────────────
# CNN switch
# ──────────────────────────────────────────────────────────
# Flip this to True when you load a convolutional network that
# expects input shaped [N, C, H, W] rather than flattened.
CNN_MODE = False           # <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<

# ──────────────────────────────────────────────────────────
# Helper functions
# ──────────────────────────────────────────────────────────
def clamp_tensor(x: torch.Tensor,
                 lower: torch.Tensor,
                 upper: torch.Tensor) -> torch.Tensor:
    """Clamp tensor values element‑wise between *lower* and *upper*."""
    return torch.max(torch.min(x, upper), lower)


def calculate_adversarial_loss(
    model: nn.Module,
    x_raw_single: torch.Tensor,
    y_single: torch.Tensor,
    eps: float = 1e-9
) -> torch.Tensor:
    """
    Black‑box cross‑entropy with ε‑floor (prevents CE == 0).

    Parameters
    ----------
    model  : nn.Module        – queried only in forward mode
    x_raw_single : [1, dim]   – raw pixels in [0,255]
    y_single     : [1]        – ground truth label
    eps          : float      – minimum CE to return
    """
    x_clip = clamp_tensor(x_raw_single, final_lower, final_upper)
    x_norm = x_clip / final_upper                       # scale to [0,1]

    if CNN_MODE:
        x_norm = x_norm.view(1, 1, *img_shape)          # keep CNN option

    # forward pass – still black‑box
    logits = model(x_norm)

    # CE for the single true class (no reduction over batch)
    log_probs = F.log_softmax(logits, dim=1)            # safe log
    ce = -log_probs[0, y_single]

    # floor to keep CE ≥ ε  (prevents fitness from flattening at 0)
    ce = torch.clamp(ce, min=eps)

    return ce



def save_uint8(img_arr: np.ndarray, path: str) -> None:
    """Round float32 array in [0,255] → uint8 and save with OpenCV."""
    img_u8 = np.clip(np.round(img_arr), final_lower_bound, final_upper_bound).astype(np.uint8)
    cv2.imwrite(path, img_u8)


# ──────────────────────────────────────────────────────────
# Load the trained model
# ──────────────────────────────────────────────────────────
MODEL_PATH = os.path.join("Models and Data splits", "MainModel_MLP1L.pt")

try:
    model = torch.jit.load(MODEL_PATH, map_location=device)
    model.to(device).eval()
    print(f"Model loaded successfully from {MODEL_PATH}")
except Exception as e:
    print(f"Error loading model from {MODEL_PATH}: {e}")
    model = None        # Guard in main()

# ──────────────────────────────────────────────────────────
# Cuckoo Search (single sample)
# ──────────────────────────────────────────────────────────
class CuckooSearchSingleSample:
    """
    Cuckoo‑Search meta‑heuristic to craft an adversarial perturbation
    for **one** image sample.
    """

    # --- constructor -------------------------------------------------
    def __init__(self,
                 n_nests: int,
                 n_iterations: int,
                 dim: int,
                 final_lower: torch.Tensor,
                 final_upper: torch.Tensor,
                 raw_lower: torch.Tensor,
                 raw_upper: torch.Tensor,
                 model: nn.Module,
                 X_sample: torch.Tensor,
                 y_sample: torch.Tensor,
                 p_a: float = 0.5,
                 lambda_param: float = 1e9,
                 step_size: float = 2.0,
                 step_decay: float = 0.98):
        # Public params
        self.n_nests        = n_nests
        self.n_iterations   = n_iterations
        self.dim            = dim
        self.final_lower    = final_lower
        self.final_upper    = final_upper
        self.raw_lower      = raw_lower
        self.raw_upper      = raw_upper
        self.model          = model
        self.X              = X_sample.unsqueeze(0)   # [1, dim]
        self.y              = y_sample                # [1]
        self.p_a            = p_a
        self.lambda_param   = lambda_param
        self.step_size      = step_size
        self.step_decay     = step_decay

        # --- population initialisation ------------------------------
        self.nests = torch.zeros((n_nests, dim), dtype=torch.float32, device=device)

        self.fitness = torch.empty(n_nests, dtype=torch.float32, device=device)
        for i in range(n_nests):
            self.fitness[i] = self._fitness(self.nests[i])

        best_idx = torch.argmin(self.fitness)
        self.best_nest    = self.nests[best_idx].clone()
        self.best_fitness = self.fitness[best_idx].item()

    # --- internals ---------------------------------------------------

    def _cauchy_flight(self) -> torch.Tensor:
        loc = torch.tensor(0.0, device=device)
        scale = torch.tensor(self.step_size, device=device)
        dist = torch.distributions.Cauchy(loc, scale)
        return dist.sample((self.dim,))

    def _new_solution(self, curr_perturb: torch.Tensor) -> torch.Tensor:
        flight = self._cauchy_flight()                              # [dim]
        cand   = clamp_tensor(curr_perturb + flight,
                              self.raw_lower, self.raw_upper)
        return cand

    def _fitness(self, perturb: torch.Tensor) -> float:
        """Fitness = ‖δ‖₂ − λ · CrossEntropy (untargeted)."""
        p_clip        = clamp_tensor(perturb, self.raw_lower, self.raw_upper)
        perturbed_raw = self.X + p_clip.unsqueeze(0)               # [1, dim]
        loss          = calculate_adversarial_loss(self.model,
                                                   perturbed_raw,
                                                   self.y).item()
        magnitude     = torch.norm(p_clip).item()
        return magnitude - self.lambda_param * loss

    # --- public API --------------------------------------------------
    def optimize(self) -> None:
        """Run Cuckoo‑Search for *n_iterations*."""
        for it in range(1, self.n_iterations + 1):
            # 1) generate new solutions
            for i in range(self.n_nests):
                cand      = self._new_solution(self.nests[i])
                cand_fit  = self._fitness(cand)

                if cand_fit < self.fitness[i]:
                    self.nests[i]  = cand
                    self.fitness[i]= cand_fit
                    if cand_fit < self.best_fitness:
                        self.best_fitness = cand_fit
                        self.best_nest    = cand.clone()

            # 2) abandon a fraction p_a of worst nests
            k = int(self.n_nests * self.p_a)
            if k > 0:
                worst_idx = torch.argsort(self.fitness)[-k:]
                for idx in worst_idx:
                    new_rand = torch.empty((self.dim,), dtype=torch.float32, device=device)\
                                  .uniform_(float(self.raw_lower),
                                            float(self.raw_upper))
                    self.nests[idx]  = new_rand
                    self.fitness[idx]= self._fitness(new_rand)

            if it in {1, self.n_iterations} or it % 10 == 0:
                print(f"    iter {it}/{self.n_iterations} — best fitness {self.best_fitness:.5f}")

            # 3) step‑size schedule (optional)
            self.step_size *= self.step_decay

    def get_best_solution(self) -> torch.Tensor:
        """Return δ*  (shape [dim])."""
        return self.best_nest

# ──────────────────────────────────────────────────────────
# Main driver – sample‑by‑sample attack
# ──────────────────────────────────────────────────────────
def main() -> None:
    if model is None:
        print("Model not loaded; aborting.")
        return

    tic = time.time()
    print("Starting sample‑by‑sample adversarial generation …")

    # ---- load data --------------------------------------------------
    data_path = os.path.join("Models and Data splits", "MLP1L_500_samples_train.pkl")
    if not os.path.exists(data_path):
        print(f"Data file not found: {data_path}")
        return

    X_full_np, y_full_np = joblib.load(data_path)
    X_full = torch.tensor(X_full_np, dtype=torch.float32)
    y_full = torch.tensor(y_full_np, dtype=torch.long)
    print(f"Loaded X: {X_full.shape}, y: {y_full.shape}")

    # ---- output dirs -----------------------------------------------
    base_dir = "Generated Data/ZC-CSA_images"
    orig_dir = os.path.join(base_dir, "Original")
    adv_dir  = os.path.join(base_dir, "Adversarial")
    pert_dir = os.path.join(base_dir, "Perturbations")
    for d in (orig_dir, adv_dir, pert_dir):
        os.makedirs(d, exist_ok=True)

    # ---- hyper‑parameters ------------------------------------------
    n_nests       = 58
    n_iterations  = 100
    p_a           = 0.5
    lambda_param  = 1e10
    step_size     = 1.0
    step_decay    = 0.98      # optional (1.0 disables decay)

    all_results = []
    processed   = 0
    adv_success = 0

    # ---- loop over digits ------------------------------------------
    for digit in range(10):     # process all digits now
        idxs = (y_full == digit).nonzero(as_tuple=True)[0]
        if idxs.numel() == 0:
            print(f"\nNo samples for digit {digit}; skipping.")
            continue

        print(f"\n── Processing digit {digit} ({idxs.numel()} samples) ──")
        X_class = X_full[idxs].to(device)
        y_class = y_full[idxs].to(device)

        for i in range(X_class.size(0)):
            X_sample = X_class[i]                      # [dim]
            y_sample = y_class[i].unsqueeze(0)         # [1]
            gidx     = int(idxs[i])

            print(f"  sample {i+1}/{X_class.size(0)} (global {gidx})")

            # -- save original --------------------------------------
            save_uint8(X_sample.cpu().numpy().reshape(img_shape),
                       os.path.join(orig_dir, f"{gidx}_lbl{digit}.png"))

            # -- Cuckoo‑Search optimiser ---------------------------
            css = CuckooSearchSingleSample(
                n_nests      = n_nests,
                n_iterations = n_iterations,
                dim          = dim,
                final_lower  = final_lower,
                final_upper  = final_upper,
                raw_lower    = raw_lower,
                raw_upper    = raw_upper,
                model        = model,
                X_sample     = X_sample,
                y_sample     = y_sample,
                p_a          = p_a,
                lambda_param = lambda_param,
                step_size    = step_size,
                step_decay   = step_decay,
            )
            css.optimize()
            delta_best = css.get_best_solution()

            # -- apply δ* ------------------------------------------
            adv_raw  = clamp_tensor(X_sample + delta_best, final_lower, final_upper)
            adv_norm = adv_raw.unsqueeze(0) / final_upper
            if CNN_MODE:
                adv_norm = adv_norm.view(1, 1, *img_shape)

            with torch.no_grad():
                logits = model(adv_norm)
                pred   = torch.argmax(logits, dim=1)

            norm_l2  = torch.norm(delta_best).item()
            success  = (pred != y_sample).item()

            # -- save files ----------------------------------------
            delta_np = delta_best.cpu().numpy().reshape(img_shape)
            np.save(os.path.join(pert_dir, f"{gidx}_delta.npy"), delta_np)

            vis = ((delta_np - raw_lower_bound) / (raw_upper_bound - raw_lower_bound) * final_upper_bound)

            save_uint8(vis, os.path.join(pert_dir, f"{gidx}_delta_vis.png"))

            if success:
                adv_success += 1
                save_uint8(adv_raw.cpu().numpy().reshape(img_shape),
                           os.path.join(adv_dir,
                                        f"{gidx}_t{digit}_p{int(pred)}_m{norm_l2:.2f}.png"))

            # -- record row ----------------------------------------
            all_results.append(dict(
                digit=digit,
                index=gidx,
                true=digit,
                pred=int(pred),
                misclassified=success,
                perturb_norm_l2=norm_l2,
            ))

            processed += 1

    # ---- export CSV summary ----------------------------------------
    df = pd.DataFrame(all_results)
    csv_path = os.path.join(base_dir, "zc_csa_500x50_results.csv")
    df.to_csv(csv_path, index=False)

    # ---- per‑digit statistics --------------------------------------
    stats = (df.groupby(["digit", "misclassified"])
               .size()
               .unstack(fill_value=0)
               .rename(columns={False: "correct", True: "misclassified"}))

    # Ensure both columns always exist
    if "misclassified" not in stats.columns:
        stats["misclassified"] = 0
    if "correct" not in stats.columns:
        stats["correct"] = 0

    stats["total"]      = stats["correct"] + stats["misclassified"]
    stats["correct_%"]  = (stats["correct"]       / stats["total"] * 100).round(2)
    stats["miscls_%"]   = (stats["misclassified"] / stats["total"] * 100).round(2)


    stats_csv = os.path.join(base_dir, "zc_csa_stats_per_digit.csv")
    stats.to_csv(stats_csv)

    toc = time.time()

    # ---- pretty print ---------------------------------------------
    print("\n===== SUMMARY =====")
    print(f"Processed samples        : {processed}")
    print(f"Successful attacks       : {adv_success}")
    print(f"Detailed results  CSV    : {csv_path}")
    print(f"Per‑digit stats   CSV    : {stats_csv}")
    print(f"Total time               : {toc - tic:.2f} s\n")

    print(stats.to_string())


if __name__ == "__main__":
    main()
'''